In [0]:
# ===========================================
# Notebook Name:
# 00_build_card_usage
#
# Purpose:
# Build card usage metrics from Silver deck data.
#
# Input:
# pokemon.silver.tournament_results
# pokemon.silver.deck_cards
#
# Output:
# pokemon.gold.card_usage
#
# Business rules:
# - Basic Energy cards are excluded
# - Special Energy cards remain included
# - Usage rate is based on distinct decks
# ===========================================

In [0]:
from pyspark.sql import functions as F

TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)

DECK_CARDS_TABLE = (
    "pokemon.silver.deck_cards"
)

CARD_USAGE_TABLE = (
    "pokemon.gold.card_usage"
)

print("Input :", TOURNAMENT_RESULTS_TABLE)
print("Input :", DECK_CARDS_TABLE)
print("Output:", CARD_USAGE_TABLE)

In [0]:
tournament_results_df = spark.table(
    TOURNAMENT_RESULTS_TABLE
)

deck_cards_df = spark.table(
    DECK_CARDS_TABLE
)

print(
    "Tournament results:",
    tournament_results_df.count(),
)

print(
    "Deck card rows:",
    deck_cards_df.count(),
)

In [0]:
eligible_decks_df = (
    tournament_results_df
    .select(
        "deck_code",
    )
    .filter(
        F.col("deck_code").isNotNull()
    )
    .dropDuplicates()
)

total_deck_count = (
    eligible_decks_df.count()
)

print(
    "Eligible deck count:",
    total_deck_count,
)

In [0]:
analysis_cards_df = (
    deck_cards_df
    .join(
        eligible_decks_df,
        on="deck_code",
        how="inner",
    )
    .filter(
        ~(
            (F.col("category") == "energy")
            & F.col("card_name").startswith(
                "基本"
            )
        )
    )
)

display(
    analysis_cards_df
    .select(
        "deck_code",
        "category",
        "card_name",
        "quantity",
    )
    .orderBy(
        "deck_code",
        "category",
        "card_name",
    )
)

In [0]:
basic_energy_count = (
    deck_cards_df
    .filter(
        (F.col("category") == "energy")
        & F.col("card_name").startswith(
            "基本"
        )
    )
    .count()
)

print(
    "Excluded basic-energy rows:",
    basic_energy_count,
)

In [0]:
card_usage_df = (
    analysis_cards_df
    .groupBy(
        "card_name",
        "category",
    )
    .agg(
        F.countDistinct(
            "deck_code"
        ).alias(
            "decks_using_card"
        ),
        F.sum(
            "quantity"
        ).alias(
            "total_copies"
        ),
        F.avg(
            "quantity"
        ).alias(
            "avg_copies_when_used"
        ),
        F.min(
            "quantity"
        ).alias(
            "min_copies_when_used"
        ),
        F.max(
            "quantity"
        ).alias(
            "max_copies_when_used"
        ),
    )
    .withColumn(
        "total_decks",
        F.lit(total_deck_count),
    )
    .withColumn(
        "inclusion_rate",
        F.col("decks_using_card")
        / F.col("total_decks"),
    )
    .withColumn(
        "inclusion_rate_pct",
        F.round(
            F.col("inclusion_rate") * 100,
            2,
        ),
    )
    .withColumn(
        "avg_copies_when_used",
        F.round(
            F.col("avg_copies_when_used"),
            2,
        ),
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

In [0]:
display(
    card_usage_df
    .orderBy(
        F.col(
            "decks_using_card"
        ).desc(),
        F.col(
            "total_copies"
        ).desc(),
    )
)

In [0]:
invalid_usage_df = (
    card_usage_df
    .filter(
        (F.col("inclusion_rate") < 0)
        | (F.col("inclusion_rate") > 1)
        | (
            F.col("decks_using_card")
            > F.col("total_decks")
        )
    )
)

invalid_count = invalid_usage_df.count()

if invalid_count > 0:
    display(invalid_usage_df)

    raise ValueError(
        f"{invalid_count} invalid usage rows"
    )

print(
    "Validation passed: "
    "all inclusion rates are valid"
)

In [0]:
remaining_basic_energy_count = (
    card_usage_df
    .filter(
        (F.col("category") == "energy")
        & F.col("card_name").startswith(
            "基本"
        )
    )
    .count()
)

if remaining_basic_energy_count > 0:
    raise ValueError(
        "Basic Energy cards remain "
        "in card_usage"
    )

print(
    "Validation passed: "
    "Basic Energy cards excluded"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CARD_USAGE_TABLE}
(
    card_name STRING NOT NULL,
    category STRING NOT NULL,
    decks_using_card BIGINT,
    total_copies BIGINT,
    avg_copies_when_used DOUBLE,
    min_copies_when_used INT,
    max_copies_when_used INT,
    total_decks INT,
    inclusion_rate DOUBLE,
    inclusion_rate_pct DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Card usage metrics excluding Basic Energy cards'
""")

In [0]:
spark.sql(
    f"TRUNCATE TABLE {CARD_USAGE_TABLE}"
)

print(
    "Existing card_usage rows removed."
)

In [0]:
(
    card_usage_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        CARD_USAGE_TABLE
    )
)

print(
    "Gold card_usage table rebuilt."
)

In [0]:
display(
    spark.table(
        CARD_USAGE_TABLE
    )
    .orderBy(
        F.col(
            "inclusion_rate"
        ).desc(),
        F.col(
            "total_copies"
        ).desc(),
    )
)

In [0]:
display(
    spark.table(
        CARD_USAGE_TABLE
    )
    .select(
        "card_name",
        "category",
        "decks_using_card",
        "total_decks",
        "inclusion_rate_pct",
        "total_copies",
        "avg_copies_when_used",
    )
    .orderBy(
        F.col(
            "inclusion_rate_pct"
        ).desc(),
        F.col(
            "total_copies"
        ).desc(),
    )
    .limit(20)
)

In [0]:
%sql

SELECT
    card_name,
    category,
    decks_using_card,
    total_decks,
    inclusion_rate_pct,
    total_copies,
    avg_copies_when_used
FROM pokemon.gold.card_usage
ORDER BY
    inclusion_rate_pct DESC,
    total_copies DESC
LIMIT 20;